### A small notebook to test functions from the simulation directory

In [1]:
import numpy as np
from states import sample_next_state, validate_transition_matrix, transition_probabilities
from scipy.special import softmax

In [2]:
BASE_TRANSITION = np.array(
    [
        [0.94, 0.05, 0.01],
        [0.10, 0.75, 0.15],
        [0.03, 0.17, 0.80],
    ],
    dtype=float,
)


In [3]:
validate_transition_matrix(BASE_TRANSITION)

In [4]:
bad_matrix = np.array(
    [
        [0.8, 0.1, 0.1],
        [0.5, 0.5, 0.5],
        [0.1, 0.2, 0.7],
    ]
)

In [5]:
validate_transition_matrix(bad_matrix)

ValueError: Each row of the transition matrix must sum to 1.

In [6]:
###cada nuevo estado modifica las probabilidades del siguiente.
###Eso crea persistencia temporal


rng = np.random.default_rng(42)

state = 0

for _ in range(20):
    state = sample_next_state(
        previous_state=state,
        transition_matrix=BASE_TRANSITION,
        rng=rng,
    )

    print(state)

0
0
0
0
0
1
1
1
1
1
1
2
2
2
2
2
2
1
1
1


In [7]:
rng = np.random.default_rng(42)

state = 0
trajectory = [state]

for _ in range(100):
    state = sample_next_state(
        previous_state=state,
        transition_matrix=BASE_TRANSITION,
        rng=rng,
    )

    trajectory.append(state)

print(trajectory)

[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]


In [8]:
unique, counts = np.unique(
    trajectory,
    return_counts=True,
)

print(dict(zip(unique, counts)))

{np.int64(0): np.int64(58), np.int64(1): np.int64(34), np.int64(2): np.int64(9)}


In [9]:
probabilities = np.array([0.94, 0.05, 0.01])

logits = np.log(probabilities)

recovered = softmax(logits)

print(logits)
print(recovered)

[-0.0618754  -2.99573227 -4.60517019]
[0.94 0.05 0.01]


In [10]:
for exposure in [0.0, 0.25, 0.50, 0.75, 1.0]:
    probabilities = transition_probabilities(
        previous_state=0,
        network_exposure=exposure,
    )

    print(
        f"Exposure={exposure:.2f}: "
        f"{np.round(probabilities, 4)}"
    )

TypeError: transition_probabilities() missing 1 required positional argument: 'network_exposure_effect'

In [11]:
NETWORK_EXPOSURE_EFFECT = np.array(
    [-1.0, 0.25, 1.25],
    dtype=float,
)

In [12]:
for exposure in [0.0, 0.25, 0.50, 0.75, 1.0]:
    probabilities = transition_probabilities(
        previous_state=0,
        network_exposure=exposure, network_exposure_effect=NETWORK_EXPOSURE_EFFECT
    )

    print(
        f"Exposure={exposure:.2f}: "
        f"{np.round(probabilities, 4)}"
    )

Exposure=0.00: [0.94 0.05 0.01]
Exposure=0.25: [0.9163 0.0666 0.0171]
Exposure=0.50: [0.8833 0.0878 0.0289]
Exposure=0.75: [0.838  0.1138 0.0482]
Exposure=1.00: [0.7772 0.1443 0.0785]


In [14]:
for previous_state in [0, 1, 2]:
    print(f"\nPrevious state = {previous_state}")

    for exposure in [0.0, 0.5, 1.0]:
        probabilities = transition_probabilities(
            previous_state=previous_state,
            network_exposure=exposure,
            network_exposure_effect=NETWORK_EXPOSURE_EFFECT
        )

        print(
            f"Exposure={exposure:.1f}: "
            f"{np.round(probabilities, 4)}"
        )


Previous state = 0
Exposure=0.0: [0.94 0.05 0.01]
Exposure=0.5: [0.8833 0.0878 0.0289]
Exposure=1.0: [0.7772 0.1443 0.0785]

Previous state = 1
Exposure=0.0: [0.1  0.75 0.15]
Exposure=0.5: [0.0509 0.7137 0.2353]
Exposure=1.0: [0.0241 0.6322 0.3437]

Previous state = 2
Exposure=0.0: [0.03 0.17 0.8 ]
Exposure=0.5: [0.0107 0.113  0.8764]
Exposure=1.0: [0.0037 0.0722 0.9241]
